# Modernizing ATS Resume Extraction: Qwen2.5-1.5B-Instruct SFT (Phase 3)
### Fine-Tuning with Unsloth + QLoRA and Exporting to GGUF `Q4_K_M` for Async Hugging Face Space Serving

**Key Technical Highlights:**
- **Base Model**: `unsloth/Qwen2.5-1.5B-Instruct` loaded in 4-bit quantization
- **Context Window**: 4096 tokens (fits full multi-page resumes)
- **LoRA Configuration**: $r = 16, \alpha = 32$, targeting all linear projections (`q, k, v, o, gate, up, down`)
- **Completion-Only Loss**: Loss is computed **strictly on the assistant's JSON response**, masking prompt tokens
- **Export**: Direct export to GGUF `Q4_K_M` (~1.0 GB file size, perfect for free-tier CPU inference via `llama.cpp`)

## Step 1: Environment & GPU Check

In [ ]:
!nvidia-smi

# Install Unsloth and modern fine-tuning libraries
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets

## Step 2: (Optional) Mount Google Drive for Checkpoint Persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/ats_qwen_checkpoints'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## Step 3: Load Qwen2.5-1.5B-Instruct in 4-bit with 4096 Context Length

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None  # None for auto detection (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## Step 4: Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapter attached successfully.")
model.print_trainable_parameters()

## Step 5: Format Data & Apply Completion-Only Masking
We use `train_on_responses_only` to compute loss **only on the assistant's JSON output**, ensuring the model strictly adheres to the JSON schema rather than memorizing noisy resume inputs.

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset

# Apply standard Qwen2.5 chat template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

# Upload your train.jsonl and val.jsonl generated in Phase 2
TRAIN_PATH = "train.jsonl"
VAL_PATH = "val.jsonl"

train_raw = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_raw = load_dataset("json", data_files=VAL_PATH, split="train")

train_dataset = train_raw.map(formatting_prompts_func, batched=True)
val_dataset = val_raw.map(formatting_prompts_func, batched=True)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

## Step 6: Training with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16
    warmup_ratio=0.03,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Mask user prompt tokens from gradient updates
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer_stats = trainer.train()
print(f"Training finished! Final loss: {trainer_stats.training_loss:.4f}")

## Step 7: Test Inference on a Sample Resume

In [ ]:
FastLanguageModel.for_inference(model)

sample_resume = """SHIVANI BHATT
Full-Stack Developer (TypeScript / Node.js / React) | Billing APIs
Canada (Remote) | +1-647-780-5215 | shivanitbhatt@gmail.com

PROFESSIONAL EXPERIENCE
Full-Stack Developer (TypeScript / Node.js / React) - Tata Consultancy Services (TCS) | Dec 2021 - Jun 2025
- Delivered high-throughput cheque fraud detection platform using microservices.

EDUCATION
Master of Computer Applications - CHARUSAT University (2014 - 2016)
"""

messages = [
    {"role": "system", "content": "You are a professional ATS resume information extraction system. Extract structured candidate details from the provided resume text into strict JSON conforming to the schema. Return ONLY valid minified JSON without markdown code blocks, preamble, or conversational text."},
    {"role": "user", "content": f"RESUME TEXT:\n{sample_resume}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=800, use_cache=True, temperature=0.0)
result_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("Extracted Model Output:")
print(result_text)

## Step 8: Export to GGUF `Q4_K_M` & Push to Hugging Face Model Hub

In [ ]:
# Save GGUF locally first
GGUF_LOCAL_PATH = "qwen2.5-1.5b-ats-gguf"
model.save_pretrained_gguf(
    GGUF_LOCAL_PATH,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Exported GGUF Q4_K_M to {GGUF_LOCAL_PATH}")

# Push GGUF directly to Hugging Face Hub
# Set your repo name and HF write token
HF_REPO_ID = "aseemlais/qwen2.5-1.5b-ats-extractor-gguf"
HF_TOKEN = "YOUR_HUGGINGFACE_WRITE_TOKEN"

if HF_TOKEN != "YOUR_HUGGINGFACE_WRITE_TOKEN":
    model.push_to_hub_gguf(
        HF_REPO_ID,
        tokenizer,
        quantization_method="q4_k_m",
        token=HF_TOKEN,
    )
    print(f"Successfully pushed GGUF model to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Please provide your HF_TOKEN to push directly to Hugging Face.")